# AB test 

A/B testing is the research method that allows you to find out the effect of a particular change in the product. The study shows which of the two versions of the product or offer gives greater effect on the selected metrics and if it is statistically significant.  

<ul>
  <li><a href="#creation-of-a-new-test-dataset-with-synthetic-data">Creation of a new test dataset with synthetic data.
  <li><a href="#ab-test">AB test.
  <li><a href="#additional-tests-in-ab-test">Additional tests in AB Test.
  <li><a href="#abn-test">ABn Test.
</ul>

In [1]:
import random

from hypex import ABTest
from hypex.dataset import Dataset, InfoRole, TargetRole, TreatmentRole
from hypex.utils import create_test_data

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


## Creation of a new test dataset with synthetic data. 

In order to be able to work with our data in HypEx, first we need to convert it into `dataset`. It is important to mark the data fields by assigning the appropriate `roles`:
- FeatureRole: a role for columns that contain features or predictor variables. Our split will be based on them. Applied by default if the role is not specified for the column.
- TreatmentRole: a role for columns that show the treatment or intervention.
- TargetRole: a role for columns that show the target or outcome variable.
- InfoRole: a role for columns that contain information about the data, such as user IDs. 

In [2]:
df=create_test_data()
df["treat"] = [random.choice([0, 1, 2]) for _ in range(len(df))]
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": TargetRole()
    }, data=df,
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,3.0,0,477.0,523.111111,47.0,M,Logistics
1,1.0,3.0,1,472.5,525.111111,61.0,M,E-commerce
2,2.0,7.0,1,479.5,479.333333,45.0,M,Logistics
3,3.0,0.0,1,489.0,419.666667,22.0,M,Logistics
4,4.0,0.0,1,483.0,401.888889,51.0,F,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,10.0,2,477.5,451.444444,18.0,M,E-commerce
9996,9996.0,2.0,1,468.5,522.555556,55.0,F,Logistics
9997,9997.0,0.0,1,478.0,402.0,40.0,M,Logistics
9998,9998.0,4.0,0,458.0,515.222222,39.0,M,Logistics


The roles' data types can be assigned automatically as shown below. Also, the fields, which were not marked, receive Feature role by default.
data["treat"] = [random.choice([0, 1, 2]) for _ in range(len(data))]
data

The roles' data types can be assigned automatically as shown below. Also, the fields, which were not marked, receive Feature role by default.

In [3]:
data.roles

{'user_id': Info(<class 'int'>),
 'treat': Treatment(<class 'int'>),
 'pre_spends': Target(<class 'float'>),
 'post_spends': Target(<class 'float'>),
 'gender': Target(<class 'object'>),
 'signup_month': Default(<class 'float'>),
 'age': Default(<class 'float'>),
 'industry': Default(<class 'object'>)}

## AB test
Then we select one of the pre-assembled pipelines, in our case `ABTest`. Also, a custom pipeline can be created based on your specific needs and requirements with custom executors.
After that we wrap our prepared `dataset` into `ExperimentData` to be able to run experiments on it and then execute the test with this data passed as the argument.

In [4]:
test = ABTest()
result = test.execute(data)

2026-08-27 10:13:19 | INFO     | hypex.experiment | ▶ Process started: GroupSizes [pandas]
2026-08-27 10:13:19 | INFO     | hypex.experiment | ✓ Process finished: GroupSizes in 0.019s
2026-08-27 10:13:19 | INFO     | hypex.experiment | ▶ Process started: OnRoleExperiment [pandas]
2026-08-27 10:13:19 | INFO     | hypex.experiment | ▶ Process started: GroupDifference [pandas]
2026-08-27 10:13:19 | INFO     | hypex.experiment | ✓ Process finished: GroupDifference in 0.012s
2026-08-27 10:13:19 | INFO     | hypex.experiment | ▶ Process started: TTest [pandas]
/Users/danilsamsutdinov/HypEx/hypex/dataset/backends/pandas_backend.py:1178: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
2026-08-27 10:13:19 | INFO     | hypex.experiment | ✓ Process finished: TTest in 0.018s
2026-08-27 10:13:19 | INFO     | hypex.experiment | ▶ Process started: TTest [pandas]
202

[ABAnalyzer DEBUG] p_values type  = <class 'hypex.dataset.dataset.SmallDataset'>
[ABAnalyzer DEBUG] p_values shape = (6, 1)
[ABAnalyzer DEBUG] p_values cols  = ['p-value']
[ABAnalyzer DEBUG] p_values index = [(1,), (2,), (1,), (2,), '(1,)', '(2,)']
[ABAnalyzer DEBUG] p_values data:        p-value
(1,)  0.095157
(2,)  0.794943
(1,)  0.095157
(2,)  0.794943
(1,)  0.735632
(2,)  0.665892
[MultiTest DEBUG] data shape     = (6, 1)
[MultiTest DEBUG] data columns   = ['p-value']
[MultiTest DEBUG] data index     = [(1,), (2,), (1,), (2,), '(1,)', '(2,)']
[MultiTest DEBUG] p_values shape = (6,)
[MultiTest DEBUG] len(fields)    = 6
[MultiTest DEBUG] len(tests)     = 6
[MultiTest DEBUG] type(new_pvalues[1]) = <class 'numpy.ndarray'>, len = 6
[MultiTest DEBUG] type(new_pvalues[0]) = <class 'numpy.ndarray'>, len = 6
[MultiTest DEBUG] fields = ['(1,)', '(2,)', '(1,)', '(2,)', '(1,)', '(2,)']
[MultiTest DEBUG] tests  = ['(1,)', '(2,)', '(1,)', '(2,)', '(1,)', '(2,)']
[Output._extract_by_reporters] re

In [5]:
result.resume

,feature,group,difference %,difference,test mean,control mean,GroupTTest pass,GroupTTest p-value
0,"['pre_spends', 'post_spends']",1┆pre_spends,-0.164862,-0.803566,486.612430,487.415996,NaN,NaN
1,"['pre_spends', 'post_spends']",1┆post_spends,-0.075226,-0.339302,450.704107,451.043409,NaN,NaN
2,"['pre_spends', 'post_spends']",2┆pre_spends,-0.026120,-0.127315,487.288681,487.415996,NaN,NaN
3,"['pre_spends', 'post_spends']",2┆post_spends,0.096833,0.436758,451.480167,451.043409,NaN,NaN
4,pre_spends,"(1,)",NaN,NaN,NaN,NaN,NOT OK,0.095157
5,pre_spends,"(2,)",NaN,NaN,NaN,NaN,NOT OK,0.794943
6,post_spends,"(1,)",NaN,NaN,NaN,NaN,NOT OK,0.735632
7,post_spends,"(2,)",NaN,NaN,NaN,NaN,NOT OK,0.665892


Note: HypEx automatically assumes the smallest value in the `TreatmentRole` column as the control group (typically `0`), and compares each other group (e.g. `1`, `2`) against it. Ensure treatment labels are correctly assigned.


### Experiment results
To show the report with summary of the test we run the `resume` method of the output of the experiment.

It displays the results of the test in the form of a table with the following columns:
- `feature`: name of the target feature, change of which we want to analyze.
- `group`: name of the test group we compare with the control group.
- `TTest pass`: result of the TTest, if it is significant or not.
- `TTest p-value`: p-value of the TTest shows the probability of obtaining the result when the null hypothesis is true. The lower the value the more significant the result is.
- `control mean`: the mean of the feature value across the control group.
- `test mean`: the mean of the feature value across the test group.
- `difference`: the difference between the mean of the test group and the mean of the control group.
- `difference %`: the normalized difference between the mean of the test group and the mean of the control group.

In [6]:
result.resume

,feature,group,difference %,difference,test mean,control mean,GroupTTest pass,GroupTTest p-value
0,"['pre_spends', 'post_spends']",1┆pre_spends,-0.164862,-0.803566,486.612430,487.415996,NaN,NaN
1,"['pre_spends', 'post_spends']",1┆post_spends,-0.075226,-0.339302,450.704107,451.043409,NaN,NaN
2,"['pre_spends', 'post_spends']",2┆pre_spends,-0.026120,-0.127315,487.288681,487.415996,NaN,NaN
3,"['pre_spends', 'post_spends']",2┆post_spends,0.096833,0.436758,451.480167,451.043409,NaN,NaN
4,pre_spends,"(1,)",NaN,NaN,NaN,NaN,NOT OK,0.095157
5,pre_spends,"(2,)",NaN,NaN,NaN,NaN,NOT OK,0.794943
6,post_spends,"(1,)",NaN,NaN,NaN,NaN,NOT OK,0.735632
7,post_spends,"(2,)",NaN,NaN,NaN,NaN,NOT OK,0.665892


The `TTest pass` column shows whether the difference between groups is statistically significant at the 5% level. 
- `OK` means the difference is significant (p < 0.05).
- `NOT OK` means no significant difference was found.

However, significance does not imply practical importance. Always examine the `difference` and `difference %` columns to assess business relevance.


The method sizes shows the statistics on the groups of the data.

The columns are:
- `control size`: the size of the control group.
- `test size`: the size of the test group.
- `control size %`: the share of the control group in the whole dataset.
- `test size %`: the share of the test group in the whole dataset.
- `group`: name of the test group.

In [7]:
result.sizes

,count┆treat,group
0,3307,"(1,)"
1,3344,"(2,)"
2,3349,NaN


In [8]:
result.multitest

,field,test,old p-value,new p-value,correction,rejected,group
0,"(1,)","(1,)",0.095157,0.570943,0.166667,False,"(1,)"
1,"(2,)","(2,)",0.794943,1.0,0.794943,False,"(1,)"
2,"(1,)","(1,)",0.095157,0.570943,0.166667,False,"(2,)"
3,"(2,)","(2,)",0.794943,1.0,0.794943,False,"(2,)"
4,"(1,)","(1,)",0.735632,1.0,0.735632,False,NaN
5,"(2,)","(2,)",0.665892,1.0,0.665892,False,NaN


### Multiple Testing Correction

When multiple metrics or test groups are analyzed, the chance of false positives increases. The `result.multitest` output shows corrected p-values using Holm's method (default) or Bonferroni if specified. The column `rejected` indicates whether the null hypothesis was rejected after correction.

To change correction method:
```python
test = ABTest(multitest_method="bonferroni")


## Additional tests in AB Test 

It is possible to add u-test and chi2-test in pipeline.

Use `u-test` for numeric variables that are skewed or non-normally distributed. It’s a non-parametric alternative to t-test.

Use `chi2-test` for categorical variables (e.g. gender, conversion rate). Note: t-test is not appropriate for categorical outcomes.

In [9]:
test = ABTest(additional_tests=['t-test', 'u-test', 'chi2-test'])
result = test.execute(data)

2026-08-27 10:13:25 | INFO     | hypex.experiment | ▶ Process started: GroupSizes [pandas]
2026-08-27 10:13:25 | INFO     | hypex.experiment | ✓ Process finished: GroupSizes in 0.017s
2026-08-27 10:13:25 | INFO     | hypex.experiment | ▶ Process started: OnRoleExperiment [pandas]
2026-08-27 10:13:25 | INFO     | hypex.experiment | ▶ Process started: GroupDifference [pandas]
2026-08-27 10:13:25 | INFO     | hypex.experiment | ✓ Process finished: GroupDifference in 0.012s
2026-08-27 10:13:25 | INFO     | hypex.experiment | ▶ Process started: TTest [pandas]
/Users/danilsamsutdinov/HypEx/hypex/dataset/backends/pandas_backend.py:1178: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
2026-08-27 10:13:25 | INFO     | hypex.experiment | ✓ Process finished: TTest in 0.019s
2026-08-27 10:13:25 | INFO     | hypex.experiment | ▶ Process started: UTest [pandas]
202

[ABAnalyzer DEBUG] p_values type  = <class 'hypex.dataset.dataset.SmallDataset'>
[ABAnalyzer DEBUG] p_values shape = (12, 1)
[ABAnalyzer DEBUG] p_values cols  = ['p-value']
[ABAnalyzer DEBUG] p_values index = [(1,), (2,), (1,), (2,), '(1,)', '(2,)', '(1,)', '(2,)', '(1,)', '(2,)', '(1,)', '(2,)']
[ABAnalyzer DEBUG] p_values data:        p-value
(1,)  0.095157
(2,)  0.794943
(1,)  0.095157
(2,)  0.794943
(1,)  0.735632
(2,)  0.665892
(1,)  0.083134
(2,)  0.620374
(1,)  0.083134
(2,)  0.620374
(1,)  0.842309
(2,)  0.690542
[MultiTest DEBUG] data shape     = (12, 1)
[MultiTest DEBUG] data columns   = ['p-value']
[MultiTest DEBUG] data index     = [(1,), (2,), (1,), (2,), '(1,)', '(2,)', '(1,)', '(2,)', '(1,)', '(2,)', '(1,)', '(2,)']
[MultiTest DEBUG] p_values shape = (12,)
[MultiTest DEBUG] len(fields)    = 12
[MultiTest DEBUG] len(tests)     = 12
[MultiTest DEBUG] type(new_pvalues[1]) = <class 'numpy.ndarray'>, len = 12
[MultiTest DEBUG] type(new_pvalues[0]) = <class 'numpy.ndarray'>, l

The additional columns are:
- `UTest pass`: result of the UTest, if it is significant or not.
- `UTest p-value`: p-value of the UTest shows the probability of obtaining the result when the null hypothesis is true. The lower the value the more significant the result is.
- `Chi2Test pass`: result of the Chi2Test, if it is significant or not.
- `Chi2Test p-value`: p-value of the Chi2Test shows the probability of obtaining the result when the null hypothesis is true. The lower the value the more significant the result is.

In [10]:
result.resume

,feature,group,difference %,difference,test mean,control mean,GroupTTest pass,GroupTTest p-value,GroupUTest pass,GroupUTest p-value
0,"['pre_spends', 'post_spends']",1┆pre_spends,-0.164862,-0.803566,486.612430,487.415996,NaN,NaN,NaN,NaN
1,"['pre_spends', 'post_spends']",1┆post_spends,-0.075226,-0.339302,450.704107,451.043409,NaN,NaN,NaN,NaN
2,"['pre_spends', 'post_spends']",2┆pre_spends,-0.026120,-0.127315,487.288681,487.415996,NaN,NaN,NaN,NaN
3,"['pre_spends', 'post_spends']",2┆post_spends,0.096833,0.436758,451.480167,451.043409,NaN,NaN,NaN,NaN
4,pre_spends,"(1,)",NaN,NaN,NaN,NaN,NOT OK,0.095157,NOT OK,0.083134
5,pre_spends,"(2,)",NaN,NaN,NaN,NaN,NOT OK,0.794943,NOT OK,0.620374
6,post_spends,"(1,)",NaN,NaN,NaN,NaN,NOT OK,0.735632,NOT OK,0.842309
7,post_spends,"(2,)",NaN,NaN,NaN,NaN,NOT OK,0.665892,NOT OK,0.690542


In [11]:
result.multitest

,field,test,old p-value,new p-value,correction,rejected,group
0,"(1,)","(1,)",0.095157,0.997607,0.095385,False,"(1,)"
1,"(2,)","(2,)",0.794943,1.0,0.794943,False,"(1,)"
2,"(1,)","(1,)",0.095157,0.997607,0.095385,False,"(2,)"
3,"(2,)","(2,)",0.794943,1.0,0.794943,False,"(2,)"
4,"(1,)","(1,)",0.735632,1.0,0.735632,False,"(1,)"
...,...,...,...,...,...,...,...
7,"(2,)","(2,)",0.620374,1.0,0.620374,False,"(2,)"
8,"(1,)","(1,)",0.083134,0.997607,0.083333,False,"(1,)"
9,"(2,)","(2,)",0.620374,1.0,0.620374,False,"(1,)"
10,"(1,)","(1,)",0.842309,1.0,0.842309,False,"(2,)"


In [12]:
result.sizes

,count┆treat,group
0,3307,"(1,)"
1,3344,"(2,)"
2,3349,NaN


## ABn Test 

Finally, we may run multiple ab tests with different methods.

In [13]:
test = ABTest(multitest_method="bonferroni")
result = test.execute(data)

2026-08-27 10:13:32 | INFO     | hypex.experiment | ▶ Process started: GroupSizes [pandas]
2026-08-27 10:13:32 | INFO     | hypex.experiment | ✓ Process finished: GroupSizes in 0.018s
2026-08-27 10:13:32 | INFO     | hypex.experiment | ▶ Process started: OnRoleExperiment [pandas]
2026-08-27 10:13:32 | INFO     | hypex.experiment | ▶ Process started: GroupDifference [pandas]
2026-08-27 10:13:32 | INFO     | hypex.experiment | ✓ Process finished: GroupDifference in 0.012s
2026-08-27 10:13:32 | INFO     | hypex.experiment | ▶ Process started: TTest [pandas]
/Users/danilsamsutdinov/HypEx/hypex/dataset/backends/pandas_backend.py:1178: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
2026-08-27 10:13:32 | INFO     | hypex.experiment | ✓ Process finished: TTest in 0.019s
2026-08-27 10:13:32 | INFO     | hypex.experiment | ▶ Process started: TTest [pandas]
202

[ABAnalyzer DEBUG] p_values type  = <class 'hypex.dataset.dataset.SmallDataset'>
[ABAnalyzer DEBUG] p_values shape = (6, 1)
[ABAnalyzer DEBUG] p_values cols  = ['p-value']
[ABAnalyzer DEBUG] p_values index = [(1,), (2,), (1,), (2,), '(1,)', '(2,)']
[ABAnalyzer DEBUG] p_values data:        p-value
(1,)  0.095157
(2,)  0.794943
(1,)  0.095157
(2,)  0.794943
(1,)  0.735632
(2,)  0.665892
[MultiTest DEBUG] data shape     = (6, 1)
[MultiTest DEBUG] data columns   = ['p-value']
[MultiTest DEBUG] data index     = [(1,), (2,), (1,), (2,), '(1,)', '(2,)']
[MultiTest DEBUG] p_values shape = (6,)
[MultiTest DEBUG] len(fields)    = 6
[MultiTest DEBUG] len(tests)     = 6
[MultiTest DEBUG] type(new_pvalues[1]) = <class 'numpy.ndarray'>, len = 6
[MultiTest DEBUG] type(new_pvalues[0]) = <class 'numpy.ndarray'>, len = 6
[MultiTest DEBUG] fields = ['(1,)', '(2,)', '(1,)', '(2,)', '(1,)', '(2,)']
[MultiTest DEBUG] tests  = ['(1,)', '(2,)', '(1,)', '(2,)', '(1,)', '(2,)']
[Output._extract_by_reporters] re

In [14]:
result.resume

,feature,group,difference %,difference,test mean,control mean,GroupTTest pass,GroupTTest p-value
0,"['pre_spends', 'post_spends']",1┆pre_spends,-0.164862,-0.803566,486.612430,487.415996,NaN,NaN
1,"['pre_spends', 'post_spends']",1┆post_spends,-0.075226,-0.339302,450.704107,451.043409,NaN,NaN
2,"['pre_spends', 'post_spends']",2┆pre_spends,-0.026120,-0.127315,487.288681,487.415996,NaN,NaN
3,"['pre_spends', 'post_spends']",2┆post_spends,0.096833,0.436758,451.480167,451.043409,NaN,NaN
4,pre_spends,"(1,)",NaN,NaN,NaN,NaN,NOT OK,0.095157
5,pre_spends,"(2,)",NaN,NaN,NaN,NaN,NOT OK,0.794943
6,post_spends,"(1,)",NaN,NaN,NaN,NaN,NOT OK,0.735632
7,post_spends,"(2,)",NaN,NaN,NaN,NaN,NOT OK,0.665892


In [15]:
result.sizes

,count┆treat,group
0,3307,"(1,)"
1,3344,"(2,)"
2,3349,NaN


In [16]:
result.multitest

,field,test,old p-value,new p-value,correction,rejected,group
0,"(1,)","(1,)",0.095157,0.570943,0.166667,False,"(1,)"
1,"(2,)","(2,)",0.794943,1.0,0.794943,False,"(1,)"
2,"(1,)","(1,)",0.095157,0.570943,0.166667,False,"(2,)"
3,"(2,)","(2,)",0.794943,1.0,0.794943,False,"(2,)"
4,"(1,)","(1,)",0.735632,1.0,0.735632,False,NaN
5,"(2,)","(2,)",0.665892,1.0,0.665892,False,NaN


## Advanced Variance Reduction Techniques

For improved statistical power and more sensitive A/B tests, consider using covariate adjustment methods:

### CUPED and CUPAC
**CUPED** (Controlled Experiments Using Pre-Experiment Data) and **CUPAC** (Covariate-Updated Pre-Analysis Correction) are advanced techniques that use historical data to reduce variance in your metrics, allowing you to:

- Detect smaller effects with the same sample size
- Reduce the sample size needed to detect the same effect  
- Increase statistical power of your experiments

These methods work by adjusting your target metrics using correlated historical features that are unaffected by the treatment.

**For a comprehensive guide on implementing these techniques, see the [CUPED & CUPAC Tutorial](СUPED&CUPAC.ipynb).**

Key benefits:
- **CUPED**: Simple single-covariate adjustment using linear regression
- **CUPAC**: Advanced multi-covariate adjustment with flexible model selection (linear, ridge, lasso, CatBoost)/

## Common Pitfalls and Recommendations

- Always assign correct roles: use `TreatmentRole` for group labels, `TargetRole` for outcome metrics. Missing roles may cause incorrect test logic.
- For categorical targets, avoid using `t-test`. Instead, include `chi2-test` in `additional_tests`.
- HypEx does not automatically balance groups. Ensure group sizes are roughly equal and comparable.
- Check for missing values. NaNs may silently affect metric calculation.
- If testing many metrics/groups, interpret results only after multiple testing correction.
- Use `result.sizes` to confirm group balance, and consider A/A testing to verify setup before real A/B.

